# Laboratorio 3: Detección de Anomalías con Isolation Forest

## Paso 1: Importar librerías

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix
import joblib

## Paso 2: Cargar y explorar datos (EDA)

In [ ]:
# Cargar datos de ejemplo (simulando tráfico de red)
np.random.seed(42)
n_normal = 1000
n_anomalias = 50

# Datos normales
datos_normales = pd.DataFrame({
    'duracion': np.random.normal(10, 2, n_normal),
    'bytes_enviados': np.random.normal(500, 100, n_normal),
    'paquetes': np.random.normal(20, 5, n_normal),
    'puerto_destino': np.random.choice([80, 443, 22], n_normal)
})

# Datos anómalos
datos_anomalos = pd.DataFrame({
    'duracion': np.random.normal(60, 10, n_anomalias),
    'bytes_enviados': np.random.normal(5000, 1000, n_anomalias),
    'paquetes': np.random.normal(200, 50, n_anomalias),
    'puerto_destino': np.random.choice([8080, 9090, 6667], n_anomalias)
})

# Combinar datos
datos = pd.concat([datos_normales, datos_anomalos], ignore_index=True)
datos['etiqueta'] = [0]*n_normal + [1]*n_anomalias

print(f"Datos cargados: {len(datos)} registros")
print(f"Normales: {n_normal}, Anomalías: {n_anomalias}")
datos.head()

In [ ]:
# EDA: Histogramas
fig, axes = plt.subplots(2, 2, figsize=(15, 10))
sns.histplot(datos['duracion'], bins=30, ax=axes[0,0], kde=True)
axes[0,0].set_title('Distribución de duración')
sns.histplot(datos['bytes_enviados'], bins=30, ax=axes[0,1], kde=True)
axes[0,1].set_title('Distribución de bytes enviados')
sns.histplot(datos['paquetes'], bins=30, ax=axes[1,0], kde=True)
axes[1,0].set_title('Distribución de paquetes')
sns.countplot(x='puerto_destino', data=datos, ax=axes[1,1])
axes[1,1].set_title('Distribución de puertos destino')
plt.tight_layout()
plt.show()

## Paso 3: Preprocesamiento de datos

In [ ]:
# Seleccionar características
X = datos[['duracion', 'bytes_enviados', 'paquetes', 'puerto_destino']]
y = datos['etiqueta']

# Normalizar datos
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print("Datos normalizados")

## Paso 4: Entrenar modelo Isolation Forest

In [ ]:
# Entrenar modelo
modelo = IsolationForest(contamination=0.05, random_state=42)
modelo.fit(X_scaled)

# Predecir
predicciones = modelo.predict(X_scaled)
predicciones = np.where(predicciones == -1, 1, 0)  # Convertir -1 (anomalía) a 1

print("Modelo entrenado")

## Paso 5: Evaluar modelo

In [ ]:
# Métricas
print("Matriz de confusión:")
print(confusion_matrix(y, predicciones))
print("\nReporte de clasificación:")
print(classification_report(y, predicciones))

In [ ]:
# Visualizar matriz de confusión
plt.figure(figsize=(8, 6))
sns.heatmap(confusion_matrix(y, predicciones), annot=True, fmt='d', cmap='Blues')
plt.title('Matriz de Confusión')
plt.ylabel('Etiqueta Real')
plt.xlabel('Etiqueta Predicha')
plt.show()

## Paso 6: Analizar umbrales y F1-Score

In [ ]:
# Obtener scores de anomalía
scores = modelo.decision_function(X_scaled)
datos['score'] = scores

# Probar diferentes umbrales
from sklearn.metrics import f1_score
umbrales = np.linspace(np.min(scores), np.max(scores), 100)
f1_scores = []

for umbral in umbrales:
    pred_umbral = np.where(scores < umbral, 1, 0)
    f1_scores.append(f1_score(y, pred_umbral))

# Encontrar mejor umbral
mejor_indice = np.argmax(f1_scores)
mejor_umbral = umbrales[mejor_indice]
mejor_f1 = f1_scores[mejor_indice]

print(f"Mejor umbral: {mejor_umbral:.4f}")
print(f"Mejor F1-Score: {mejor_f1:.4f}")

# Visualizar
plt.figure(figsize=(10, 6))
plt.plot(umbrales, f1_scores, label='F1-Score')
plt.scatter(mejor_umbral, mejor_f1, color='red', s=100, zorder=5, label=f'Mejor umbral ({mejor_umbral:.2f})')
plt.xlabel('Umbral')
plt.ylabel('F1-Score')
plt.title('Umbral vs F1-Score')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# Top 10 anomalías
top_anomalias = datos.sort_values('score').head(10)
print("Top 10 anomalías:")
top_anomalias

## Paso 7: Guardar modelo

In [ ]:
# Guardar modelo y scaler
joblib.dump(modelo, 'modelo_anomalias.pkl')
joblib.dump(scaler, 'scaler.pkl')
print("Modelo y scaler guardados exitosamente")